# Semana 5 — Perceptrón multicapa con Keras: dígitos manuscritos
### Cuaderno guiado (ALUMNO)

Un **perceptrón multicapa** (MLP, *multilayer perceptron*) es la red neuronal más sencilla: la entrada
pasa por una o varias capas ocultas de neuronas y termina en una capa de salida. En este cuaderno
entrenamos uno con **Keras** para reconocer dígitos del 0 al 9 en imágenes pequeñas.

Tiene cinco partes:

1. **Los datos**: cargar las imágenes, mirarlas y preparar entrenamiento, validación y test.
2. **Un MLP en Keras**: construirlo, entrenarlo, medir en qué se equivoca, darle varias capas ocultas y
   pararlo a tiempo (parada temprana).
3. **¿Cuántas neuronas?**: elegir el tamaño de la capa oculta con validación, estimar el resultado con
   el test y ver cómo se escribe la regularización (L2 y dropout).
4. **Optimizadores**: comparar SGD, SGD con momento, RMSProp y Adam.
5. **Regresión**: estimar el ángulo de giro de cada imagen con el mismo tipo de red.

Las mismas imágenes y la misma partición se usan en `Semana5_convolucionales`, donde una red
convolucional se compara con el MLP de este cuaderno. La teoría está en
`presentacion-redes-neuronales/redes_neuronales.pdf` y en `Semana5_sgdm_rmsprop_adam.pdf`.

**Cómo trabajar**

- El código está completo: ejecuta las celdas **en orden**, de arriba abajo, sin cambiar nada.
- Lee el texto y los resultados, y responde a las **preguntas** en las celdas «Tu respuesta».
- El cuaderno entrena diecisiete redes pequeñas. En un portátil reciente sin GPU tarda alrededor de un
  minuto en total; las celdas más lentas lo avisan.

## 0. Preparación

Cargamos las librerías (importar TensorFlow tarda unos segundos).

In [ ]:
%matplotlib inline
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'   # oculta los mensajes informativos de TensorFlow

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from tensorflow import keras
from tensorflow.keras import layers

## 1. Los datos

La carpeta `../datasets/DigitDataset/` tiene **10.000 imágenes** de dígitos del 0 al 9, de 28 × 28
píxeles en escala de grises, cada una girada un ángulo al azar entre −45° y 45°. Están repartidas en
subcarpetas `0/`, `1/`, …, `9/` según el dígito. Dos ficheros CSV fijan la partición: `digitTrain.csv`
lista las 5.000 imágenes de **entrenamiento** y `digitTest.csv`, las 5.000 de **test**. Cada fila tiene
el nombre del fichero (`image`), el dígito (`digit`) y el ángulo de giro (`angle`, que usamos en la sección 5).

La función `carga_digitos` lee las imágenes que lista un CSV y las apila en un array de forma
(imágenes, 28, 28).

In [ ]:
carpeta_digitos = '../datasets/DigitDataset'


def carga_digitos(nombre_csv):
    # Lee las imágenes que lista el CSV. Devuelve los píxeles, de forma (n, 28, 28), y los dígitos.
    ruta_csv = os.path.join(carpeta_digitos, nombre_csv)
    tabla = pd.read_csv(ruta_csv)
    lista_imagenes = []
    for fila in tabla.itertuples():
        subcarpeta = str(fila.digit)
        ruta_imagen = os.path.join(carpeta_digitos, subcarpeta, fila.image)
        imagen = Image.open(ruta_imagen)
        pixeles = np.array(imagen)
        lista_imagenes.append(pixeles)
    imagenes = np.stack(lista_imagenes)
    digitos = tabla['digit'].to_numpy()
    return imagenes, digitos


imagenes_train_completo, y_train_completo = carga_digitos('digitTrain.csv')
imagenes_test, y_test = carga_digitos('digitTest.csv')

print("Entrenamiento:", imagenes_train_completo.shape, "| test:", imagenes_test.shape)
print("Tipo de los píxeles:", imagenes_train_completo.dtype)
print("Valores mínimo y máximo:", imagenes_train_completo.min(), imagenes_train_completo.max())
recuento_train_completo = np.bincount(y_train_completo)
print("Imágenes de cada dígito (0 a 9) en entrenamiento:", recuento_train_completo)

Las primeras imágenes de entrenamiento, con su dígito:

In [ ]:
fig, ax = plt.subplots(2, 8, figsize=(12, 3.4))
ejes = ax.flatten()
for posicion in range(16):
    eje = ejes[posicion]
    eje.imshow(imagenes_train_completo[posicion], cmap='gray')
    eje.set_title(f'dígito {y_train_completo[posicion]}')
    eje.axis('off')
plt.tight_layout()
plt.show()

### Escalar los píxeles

Los píxeles son enteros de 0 (negro) a 255 (blanco). Las redes neuronales se entrenan mejor con
entradas pequeñas, del orden de 1, así que los dividimos entre 255 para llevarlos al intervalo [0, 1].
Dividir entre un número fijo no aprende nada de los datos, así que se aplica igual a entrenamiento y a
test sin riesgo de fuga de información.

In [ ]:
X_train_completo = imagenes_train_completo / 255
X_test = imagenes_test / 255

### Cada imagen, una fila

La red de este cuaderno, como los modelos de sklearn, recibe una tabla: una fila por ejemplo y una
columna por variable. Aquí cada ejemplo es una imagen y cada variable, un píxel. `reshape(5000, 784)`
coloca los 28 × 28 = 784 píxeles de cada imagen en una sola fila, recorriéndola fila a fila: los 28
primeros valores son la fila de arriba de la imagen, los 28 siguientes la segunda, y así hasta la de
abajo. Ningún valor cambia, solo su colocación; como la división entre 255, se aplica igual a
entrenamiento y a test. Para volver a ver una imagen basta con `reshape(28, 28)`. Ejecuta la celda sin
cambiar nada.

In [ ]:
X_train_completo = X_train_completo.reshape(5000, 784)
X_test = X_test.reshape(5000, 784)
print("Entrenamiento:", X_train_completo.shape, "| test:", X_test.shape)

### Separar una validación

Para elegir cosas como el número de neuronas necesitamos imágenes que el modelo no haya visto y que
**no** sean las de test, que se reservan para el final. Separamos 1.000 de las 5.000 imágenes de
entrenamiento como **validación**. Las imágenes no tienen orden temporal (no son horas consecutivas de
un proceso), así que la separación puede hacerse al azar. Es **estratificada** (`stratify`): las dos
partes conservan la proporción de cada dígito. `random_state=0` hace que salga siempre igual.

La referencia más simple es predecir siempre la **clase mayoritaria** del entrenamiento: cualquier
modelo útil tiene que superar su **exactitud** (*accuracy*), la fracción de imágenes que acierta.

Variables de esta sección:

| Variable | Qué contiene |
|---|---|
| `X_train_completo`, `y_train_completo` | las 5.000 imágenes de entrenamiento (escaladas, una fila de 784 píxeles cada una) y sus dígitos |
| `X_train`, `y_train` | las 4.000 con las que se entrenan las redes |
| `X_val`, `y_val` | las 1.000 de validación |
| `X_test`, `y_test` | las 5.000 de test; solo se usan al final de la sección 3 |
| `acierto_mayoritaria_val` | exactitud en validación de predecir siempre la clase mayoritaria |

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_completo, y_train_completo, test_size=1000, random_state=0, stratify=y_train_completo)
print("Entrenamiento:", X_train.shape, "| validación:", X_val.shape)

recuento_train = np.bincount(y_train)
clase_mayoritaria = np.argmax(recuento_train)   # si hay empate, np.argmax se queda con la primera
acierto_mayoritaria_val = np.mean(y_val == clase_mayoritaria)
print("Imágenes de cada dígito en entrenamiento:", recuento_train)
print(f"Clase mayoritaria: {clase_mayoritaria} | exactitud en validación si siempre la predecimos: {acierto_mayoritaria_val:.3f}")

## 2. Un MLP en Keras

Cada imagen llega a la red como una fila de 784 píxeles. La red tiene dos capas:

1. Una capa oculta `Dense` de **64 neuronas** con activación **ReLU**. Cada neurona está conectada a los
   784 píxeles.
2. Una capa de salida `Dense` de **10 neuronas**, una por dígito, con activación **softmax**, que
   convierte las 10 salidas en probabilidades que suman 1. La predicción es el dígito más probable.

Con softmax, la pérdida es la **entropía cruzada**. Keras la ofrece en dos versiones:
`categorical_crossentropy`, si las etiquetas vienen en *one-hot* (un vector de nueve ceros y un uno), y
`sparse_categorical_crossentropy`, si vienen como enteros de 0 a 9, como las nuestras.

En Keras, la red se escribe como una lista de capas dentro de `keras.Sequential`. La lista empieza con
`keras.Input(shape=(784,))`, que declara que cada ejemplo es un vector de 784 valores (sin contar el
número de imágenes), y sigue con las capas en orden, de la entrada a la salida.

Variables de esta sección:

| Variable | Qué contiene |
|---|---|
| `perdida` | nombre de la función de pérdida para Keras |
| `modelo` | el MLP de 64 neuronas |
| `historial` | pérdida y exactitud de entrenamiento y de validación en cada época |
| `prediccion_val` | dígito que predice el MLP para cada imagen de validación |
| `modelo_profundo` | MLP con dos capas ocultas, de 128 y 64 neuronas |
| `modelo_parada` | MLP de 64 neuronas entrenado con parada temprana |

In [ ]:
keras.utils.set_random_seed(0)   # fija los pesos iniciales y el orden de los lotes

modelo = keras.Sequential([
    keras.Input(shape=(784,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax'),
])

Elegimos la pérdida y compilamos la red con el optimizador Adam, pidiendo que mida también la
exactitud (`metrics=['accuracy']`). Keras no elige la pérdida por su cuenta: sin ella, `compile` no se
queja, pero `fit` se detiene con un error. `summary` muestra cada capa y su número de parámetros.

**Pérdida y métricas.** La **pérdida** es lo que el optimizador minimiza, y para eso necesita su
gradiente: la entropía cruzada cambia de forma suave con los pesos. Las **métricas**, como la exactitud,
solo se miden; no intervienen en el entrenamiento. La exactitud se entiende mejor, pero no se puede
minimizar con gradientes: solo cambia a saltos, cuando una imagen pasa de mal a bien clasificada, y
entre salto y salto su gradiente es cero. `metrics` es una lista: se pueden pedir varias métricas a la
vez, como haremos en la regresión de la sección 5.

In [ ]:
perdida = 'sparse_categorical_crossentropy'

modelo.compile(optimizer='adam', loss=perdida, metrics=['accuracy'])
modelo.summary()

**Pregunta 2.1.** ¿De dónde sale el número de parámetros (pesos y sesgos) de la capa oculta? ¿Y el de
la capa de salida? ¿Qué parte del total está en la capa oculta?

**Tu respuesta:** *escribe aquí*

### Entrenar

`fit` entrena la red durante 30 **épocas** (30 pasadas por las 4.000 imágenes de entrenamiento), en
lotes de 32 imágenes; en la sección 4 veremos qué significa esto. Con `validation_data`, al final de
cada época Keras mide la pérdida y la exactitud en validación, sin usar esas imágenes para aprender.

Lo que mide lo devuelve `fit` en un **historial**: `historial.history` es un diccionario con una lista
por clave y un valor por época. Hay una clave para la pérdida y otra para cada métrica de `compile`, y
cada una aparece dos veces: con su nombre, en entrenamiento, y con el prefijo `val_`, en validación. La
celda imprime las claves; con ellas dibujamos después las curvas. Tarda unos segundos.

In [ ]:
n_epocas = 30
historial = modelo.fit(X_train, y_train, validation_data=(X_val, y_val),
                       epochs=n_epocas, batch_size=32, verbose=0)

claves_historial = list(historial.history.keys())
print("Claves del historial:", claves_historial)

perdida_train_final = historial.history['loss'][-1]
perdida_val_final = historial.history['val_loss'][-1]
print(f"Última época: pérdida de entrenamiento {perdida_train_final:.3f}; de validación {perdida_val_final:.3f}")

In [ ]:
epocas = np.arange(1, n_epocas + 1)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(epocas, historial.history['loss'], label='entrenamiento')
ax[0].plot(epocas, historial.history['val_loss'], label='validación')
ax[0].set_ylabel('pérdida (entropía cruzada)')
ax[0].set_title('Pérdida')
ax[1].plot(epocas, historial.history['accuracy'], label='entrenamiento')
ax[1].plot(epocas, historial.history['val_accuracy'], label='validación')
ax[1].set_ylabel('exactitud')
ax[1].set_title('Exactitud')
for eje in ax:
    eje.set_xlabel('época')
    eje.grid(alpha=0.3)
    eje.legend()
plt.tight_layout()
plt.show()

### Comparar con la clase mayoritaria

Medimos la exactitud del MLP en entrenamiento y en validación, y la comparamos con la de predecir siempre
la clase mayoritaria.

In [ ]:
perdida_mlp_val, acierto_mlp_val = modelo.evaluate(X_val, y_val, verbose=0)
perdida_mlp_train, acierto_mlp_train = modelo.evaluate(X_train, y_train, verbose=0)

print("Exactitud en validación:")
print(f"  clase mayoritaria     {acierto_mayoritaria_val:.3f}")
print(f"  MLP de 64 neuronas    {acierto_mlp_val:.3f}   (en entrenamiento: {acierto_mlp_train:.3f})")

**Pregunta 2.2.** Mira las curvas: ¿cómo evolucionan la pérdida de entrenamiento y la de validación a lo
largo de las épocas? ¿Qué exactitud alcanza el MLP en entrenamiento y en validación? ¿Cuánto mejora a la
clase mayoritaria?

**Tu respuesta:** *escribe aquí*

### ¿En qué se equivoca?

La **matriz de confusión** cuenta, para cada dígito real (filas), qué dígito predijo la red (columnas);
los aciertos están en la diagonal. `predict` devuelve las 10 probabilidades de cada imagen y
`np.argmax` se queda con la posición de la mayor, que es el dígito predicho.

In [ ]:
probabilidades_val = modelo.predict(X_val, verbose=0)
prediccion_val = np.argmax(probabilidades_val, axis=1)

fig, ax = plt.subplots(figsize=(6, 5.5))
ConfusionMatrixDisplay.from_predictions(y_val, prediccion_val, ax=ax, colorbar=False)
ax.set_xlabel('dígito predicho')
ax.set_ylabel('dígito real')
ax.set_title('Matriz de confusión del MLP (validación)')
plt.tight_layout()
plt.show()

De la matriz de confusión salen las métricas por clase de la semana 3. Para cada dígito, la
**precisión** es la fracción de las imágenes que la red asigna a ese dígito que de verdad lo son, y el
***recall***, la fracción de las imágenes de ese dígito que la red reconoce; el **F1** combina las dos.
`classification_report` de sklearn las calcula a partir de las predicciones.

In [ ]:
informe_val = classification_report(y_val, prediccion_val, digits=3)
print(informe_val)

Algunas imágenes de validación mal clasificadas. Para dibujar cada una, `reshape(28, 28)` vuelve a
colocar sus 784 píxeles en 28 filas de 28:

In [ ]:
posiciones_error = np.flatnonzero(prediccion_val != y_val)

fig, ax = plt.subplots(2, 6, figsize=(10, 4))
ejes = ax.flatten()
for numero in range(12):
    posicion = posiciones_error[numero]
    eje = ejes[numero]
    imagen = X_val[posicion].reshape(28, 28)
    eje.imshow(imagen, cmap='gray')
    eje.set_title(f'real {y_val[posicion]}, predice {prediccion_val[posicion]}', fontsize=9)
    eje.axis('off')
plt.tight_layout()
plt.show()

**Pregunta 2.3.** ¿Qué dígitos confunde más el MLP? Mirando las imágenes mal clasificadas, ¿te parecen
errores comprensibles? ¿Una persona se equivocaría con alguna?

**Tu respuesta:** *escribe aquí*

### Varias capas ocultas

Una red densa puede tener más de una capa oculta: se ponen más capas `Dense` en la lista, una detrás de
otra, y cada una recibe las salidas de la anterior. Esta red tiene dos capas ocultas con ReLU, de 128 y
de 64 neuronas, y la misma capa de salida softmax de 10 neuronas.

In [ ]:
keras.utils.set_random_seed(0)

modelo_profundo = keras.Sequential([
    keras.Input(shape=(784,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax'),
])
modelo_profundo.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
modelo_profundo.summary()

La entrenamos igual que el MLP de 64 neuronas y comparamos la exactitud en validación. Tarda unos
segundos.

In [ ]:
modelo_profundo.fit(X_train, y_train, epochs=n_epocas, batch_size=32, verbose=0)
perdida_profundo_val, acierto_profundo_val = modelo_profundo.evaluate(X_val, y_val, verbose=0)

print("Exactitud en validación:")
print(f"  MLP de 64 neuronas            {acierto_mlp_val:.3f}")
print(f"  MLP de dos capas (128 y 64)   {acierto_profundo_val:.3f}")

### Parada temprana

Hasta aquí hemos entrenado 30 épocas, una cifra elegida sin más. En lugar de adivinar cuántas épocas
hacen falta, se puede dejar que lo decida la validación con la **parada temprana** (`EarlyStopping`). Al
final de cada época mide la pérdida de validación; si pasan `patience` épocas seguidas sin mejorar la
mejor conseguida, detiene el entrenamiento, y con `restore_best_weights=True` la red recupera los pesos
de su mejor época. Se le da un máximo de épocas holgado (`epochs=200`) para que sea ella quien decida
cuándo parar. Se pasa a `fit` en la lista `callbacks`: funciones que Keras llama durante el
entrenamiento.

Entrenamos otra vez la red de 64 neuronas, igual que al principio de la sección, pero con parada temprana
y paciencia 10. Tarda unos segundos.

In [ ]:
keras.utils.set_random_seed(0)

modelo_parada = keras.Sequential([
    keras.Input(shape=(784,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax'),
])
modelo_parada.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

parada = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
historial_parada = modelo_parada.fit(X_train, y_train, validation_data=(X_val, y_val),
                                     epochs=200, batch_size=32, callbacks=[parada], verbose=0)

n_epocas_parada = len(historial_parada.history['loss'])
perdidas_val_parada = historial_parada.history['val_loss']
posicion_mejor_epoca = np.argmin(perdidas_val_parada)
mejor_epoca = posicion_mejor_epoca + 1
perdida_parada_val, acierto_parada_val = modelo_parada.evaluate(X_val, y_val, verbose=0)

print(f"Se detuvo en la época {n_epocas_parada}; la mejor pérdida de validación fue la de la época {mejor_epoca}.")
print("Exactitud en validación:")
print(f"  30 épocas fijas     {acierto_mlp_val:.3f}")
print(f"  parada temprana     {acierto_parada_val:.3f}")

## 3. ¿Cuántas neuronas?

El número de neuronas de la capa oculta es un **hiperparámetro**: la red no lo aprende, lo elegimos
nosotros. Probamos cinco tamaños, de 2 a 512 neuronas, y elegimos con la **validación**. Todas las redes
se entrenan como en la sección 2: Adam, 30 épocas, lotes de 32 y la misma semilla, para que de una red a
otra solo cambie el tamaño. Usamos 30 épocas fijas, sin parada temprana, porque al final reentrenaremos
el tamaño elegido con entrenamiento y validación juntos, y ya no quedará validación con la que decidir
cuándo parar.

La función `crea_mlp` construye y compila un MLP como el de la sección 2, con el número de neuronas y el
optimizador que se le pasen.

Variables de esta sección:

| Variable | Qué contiene |
|---|---|
| `tamanos` | números de neuronas de la capa oculta que probamos |
| `resultados_tamano` | exactitud de entrenamiento y de validación de cada tamaño |
| `historiales_tamano` | historial de entrenamiento de cada tamaño |
| `n_neuronas_elegido` | el tamaño con mejor exactitud en validación |
| `modelo_final` | MLP del tamaño elegido, reentrenado con las 5.000 imágenes |
| `modelo_l2`, `modelo_dropout` | la red de 512 neuronas con regularización L2 y con dropout |

In [ ]:
def crea_mlp(n_neuronas, optimizador):
    # MLP para imágenes en filas de 784 píxeles: capa oculta ReLU de n_neuronas y salida softmax de 10.
    modelo_nuevo = keras.Sequential([
        keras.Input(shape=(784,)),
        layers.Dense(n_neuronas, activation='relu'),
        layers.Dense(10, activation='softmax'),
    ])
    modelo_nuevo.compile(optimizer=optimizador, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return modelo_nuevo

El bucle entrena las cinco redes, una por tamaño, y guarda la exactitud de cada una en entrenamiento y
en validación. **Tarda menos de medio minuto.**

In [ ]:
tamanos = [2, 8, 32, 128, 512]

filas_resultados = []
historiales_tamano = {}

for n_neuronas in tamanos:
    keras.utils.set_random_seed(0)   # misma semilla para todas: solo cambia el tamaño
    modelo_tamano = crea_mlp(n_neuronas, 'adam')
    historial_tamano = modelo_tamano.fit(X_train, y_train, validation_data=(X_val, y_val),
                                         epochs=n_epocas, batch_size=32, verbose=0)
    perdida_train, acierto_train = modelo_tamano.evaluate(X_train, y_train, verbose=0)

    perdida_val, acierto_val = modelo_tamano.evaluate(X_val, y_val, verbose=0)

    fila = {'neuronas': n_neuronas, 'acierto_train': acierto_train, 'acierto_val': acierto_val}
    filas_resultados.append(fila)
    historiales_tamano[n_neuronas] = historial_tamano.history
    print(f"{n_neuronas} neuronas: entrenada")

resultados_tamano = pd.DataFrame(filas_resultados)

La exactitud de cada tamaño en entrenamiento y en validación:

In [ ]:
tabla_tamanos = resultados_tamano.set_index('neuronas')
tabla_tamanos.round(3)

In [ ]:
acierto_train_tamano = resultados_tamano['acierto_train']
acierto_val_tamano = resultados_tamano['acierto_val']

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(tamanos, acierto_val_tamano, 'o-', label='validación')
ax.plot(tamanos, acierto_train_tamano, 's--', label='entrenamiento')
ax.axhline(acierto_mayoritaria_val, color='black', ls=':', label='clase mayoritaria (validación)')
ax.set_xscale('log', base=2)
ax.set_xticks(tamanos)
ax.set_xticklabels(tamanos)
ax.set_xlabel('neuronas en la capa oculta (escala logarítmica)')
ax.set_ylabel('exactitud')
ax.set_title('Exactitud según el tamaño de la capa oculta')
ax.grid(alpha=0.3)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

Las curvas de entrenamiento de cada tamaño. Línea discontinua: entrenamiento; línea continua:
validación. La pérdida está en escala logarítmica para que se vean a la vez valores muy distintos.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for numero_tamano in range(len(tamanos)):
    n_neuronas = tamanos[numero_tamano]
    historia = historiales_tamano[n_neuronas]
    color = f'C{numero_tamano}'
    ax[0].plot(epocas, historia['loss'], ls='--', color=color)
    ax[0].plot(epocas, historia['val_loss'], color=color, label=f'{n_neuronas} neuronas')
    ax[1].plot(epocas, historia['accuracy'], ls='--', color=color)
    ax[1].plot(epocas, historia['val_accuracy'], color=color, label=f'{n_neuronas} neuronas')
ax[0].set_yscale('log')
ax[0].set_ylabel('pérdida (escala logarítmica)')
ax[0].set_title('Pérdida: entrenamiento (- -) y validación (—)')
ax[1].set_ylabel('exactitud')
ax[1].set_title('Exactitud: entrenamiento (- -) y validación (—)')
for eje in ax:
    eje.set_xlabel('época')
    eje.grid(alpha=0.3)
    eje.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Pregunta 3.1.** ¿Con qué tamaños ves **subentrenamiento**? ¿Y **sobreentrenamiento**? Justifícalo con
las curvas y con la diferencia entre la exactitud de entrenamiento y la de validación.

**Tu respuesta:** *escribe aquí*

### Elegir el tamaño y estimar con el test

Nos quedamos con el tamaño de mayor exactitud en validación. Después lo **reentrenamos** con las
5.000 imágenes de entrenamiento (entrenamiento + validación): la validación ya ha servido para elegir, y
con más datos el modelo suele salir algo mejor. Usamos las mismas 30 épocas y la semilla 0.

Por último, medimos la exactitud en el **test**, una sola vez, con este modelo final, y como referencia
la de la clase mayoritaria. Tarda unos segundos.

In [ ]:
columna_val = resultados_tamano['acierto_val']
valores_val = columna_val.to_numpy()
posicion_mejor = np.argmax(valores_val)
n_neuronas_elegido = tamanos[posicion_mejor]
print("Tamaño elegido:", n_neuronas_elegido, "neuronas")

In [ ]:
keras.utils.set_random_seed(0)
modelo_final = crea_mlp(n_neuronas_elegido, 'adam')
modelo_final.fit(X_train_completo, y_train_completo, epochs=n_epocas, batch_size=32, verbose=0)

perdida_test, acierto_test = modelo_final.evaluate(X_test, y_test, verbose=0)
clase_mayoritaria_completo = np.argmax(recuento_train_completo)
acierto_mayoritaria_test = np.mean(y_test == clase_mayoritaria_completo)

print("Exactitud en test (5.000 imágenes, medida una sola vez):")
print(f"  clase mayoritaria     {acierto_mayoritaria_test:.3f}")
print(f"  MLP de {n_neuronas_elegido} neuronas   {acierto_test:.3f}")

**Pregunta 3.2.** ¿Qué tamaño ha salido elegido? Compara la exactitud de test con la de validación de
ese tamaño. ¿Por qué no se debe volver a elegir el tamaño mirando el test?

**Tu respuesta:** *escribe aquí*

### Regularización

La **regularización** frena el sobreentrenamiento limitando lo que la red puede memorizar. Vemos cómo se
escriben en Keras dos formas de hacerlo, con la red de 512 neuronas y el mismo entrenamiento de esta
sección:

- **L2**: suma a la pérdida λ veces la suma de los cuadrados de los pesos de la capa, como el *ridge* de
  la semana 2, para que los pesos no crezcan demasiado. Se indica en la capa con
  `kernel_regularizer=keras.regularizers.l2(λ)`; usamos λ = 0,001.
- **Dropout**: mientras se entrena, en cada lote apaga al azar una fracción de las neuronas de la capa
  anterior (aquí, la mitad), para que la red no dependa de ninguna en concreto. Es una capa más,
  `layers.Dropout(0.5)`, que solo actúa al entrenar: al predecir se usan todas las neuronas.

La parada temprana de la sección 2 es otra forma de regularización. Las dos celdas tardan unos
segundos.

In [ ]:
keras.utils.set_random_seed(0)

modelo_l2 = keras.Sequential([
    keras.Input(shape=(784,)),
    layers.Dense(512, activation='relu', kernel_regularizer=keras.regularizers.l2(0.001)),
    layers.Dense(10, activation='softmax'),
])
modelo_l2.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
modelo_l2.fit(X_train, y_train, epochs=n_epocas, batch_size=32, verbose=0)
perdida_l2_val, acierto_l2_val = modelo_l2.evaluate(X_val, y_val, verbose=0)
print(f"Red de 512 neuronas con L2: exactitud en validación = {acierto_l2_val:.3f}")

In [ ]:
keras.utils.set_random_seed(0)

modelo_dropout = keras.Sequential([
    keras.Input(shape=(784,)),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax'),
])
modelo_dropout.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
modelo_dropout.fit(X_train, y_train, epochs=n_epocas, batch_size=32, verbose=0)
perdida_dropout_val, acierto_dropout_val = modelo_dropout.evaluate(X_val, y_val, verbose=0)

acierto_512_val = tabla_tamanos.loc[512, 'acierto_val']
print("Exactitud en validación de la red de 512 neuronas:")
print(f"  sin regularización   {acierto_512_val:.3f}")
print(f"  con L2               {acierto_l2_val:.3f}")
print(f"  con dropout          {acierto_dropout_val:.3f}")

## 4. Optimizadores

### Épocas y actualizaciones

Hasta ahora hemos entrenado 30 **épocas** con **lotes** de 32 imágenes (`batch_size=32`). Una época no
es lo mismo que una actualización de los pesos:

- Con cada **lote**, Keras calcula la pérdida media de esas 32 imágenes y su gradiente, y **actualiza**
  los pesos una vez: un paso del optimizador.
- Una **época** es una pasada completa por las imágenes de entrenamiento: todos los lotes, uno tras
  otro (Keras los baraja al empezar cada época).

Con n imágenes y lotes de tamaño B hay n/B actualizaciones por época, redondeando hacia arriba: si n no
es múltiplo de B, el último lote es más pequeño.

| Variable | Qué contiene |
|---|---|
| `n_train` | número de imágenes de entrenamiento |
| `tamano_lote` | tamaño de cada lote, B |
| `actualizaciones_por_epoca` | cuántas veces se actualizan los pesos en cada época |

In [ ]:
n_train = len(y_train)
tamano_lote = 32

actualizaciones_por_epoca = int(np.ceil(n_train / tamano_lote))

**Pregunta 4.1.** ¿Cuántas veces actualiza sus pesos cada red en las 30 épocas? ¿Y si los lotes fueran
de 128 imágenes? Si comparas dos entrenamientos con distinto tamaño de lote, ¿por qué no basta con
compararlos época a época?

**Tu respuesta:** *escribe aquí*

### Cuatro optimizadores

Los cuatro usan el gradiente del lote para actualizar los pesos, pero de forma distinta (las ecuaciones
están en `Semana5_sgdm_rmsprop_adam.pdf`):

- **SGD**: da un paso proporcional al gradiente, w ← w − α·g, donde α es la **tasa de aprendizaje**.
- **SGD con momento**: acumula los gradientes anteriores en una «velocidad» y avanza según ella. El
  coeficiente de momento (aquí, 0,9) dice cuánto pesa lo acumulado.
- **RMSProp**: divide el paso de cada peso por la raíz de una media móvil de sus gradientes al cuadrado,
  así que cada peso tiene su propio tamaño de paso.
- **Adam**: combina las dos ideas, el momento y la división de RMSProp.

**Tasa de aprendizaje de cada uno.** Usamos la que Keras trae por defecto: **α = 0,01** para SGD, con y
sin momento, y **α = 0,001** para RMSProp y Adam. Son los valores con los que se suele empezar. Usar la
misma α para todos no sería más justo, porque no significa lo mismo: en SGD el paso es α por el
gradiente, mientras que RMSProp y Adam dividen el gradiente por su tamaño típico, de modo que α es
directamente el tamaño aproximado del paso de cada peso.

Usamos la red de la sección 2 (64 neuronas), 30 épocas, lotes de 32 y la misma semilla. Aquí las 30 épocas fijas, sin parada temprana, son necesarias: comparamos
cuánto aprende cada optimizador en el mismo número de épocas. Un optimizador
guarda su propio estado y no se puede compartir entre redes, así que la función `crea_optimizador` crea
uno nuevo cada vez.

In [ ]:
def crea_optimizador(nombre):
    # Devuelve un optimizador nuevo, con la tasa de aprendizaje por defecto de Keras.
    if nombre == 'SGD':
        optimizador = keras.optimizers.SGD(learning_rate=0.01)
    elif nombre == 'SGD con momento':
        optimizador = keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)
    elif nombre == 'RMSProp':
        optimizador = keras.optimizers.RMSprop(learning_rate=0.001)
    elif nombre == 'Adam':
        optimizador = keras.optimizers.Adam(learning_rate=0.001)
    return optimizador

El bucle entrena las cuatro redes, una por optimizador, y guarda, en cada época, la pérdida de
entrenamiento y la exactitud de validación. **Tarda unos segundos.**

In [ ]:
nombres_optimizadores = ['SGD', 'SGD con momento', 'RMSProp', 'Adam']
perdidas_por_optimizador = {}
aciertos_val_por_optimizador = {}

for nombre in nombres_optimizadores:
    keras.utils.set_random_seed(0)   # misma semilla para todas: solo cambia el optimizador
    optimizador = crea_optimizador(nombre)
    modelo_opt = crea_mlp(64, optimizador)
    historial_opt = modelo_opt.fit(X_train, y_train, validation_data=(X_val, y_val),
                                   epochs=n_epocas, batch_size=tamano_lote, verbose=0)
    perdidas_por_optimizador[nombre] = historial_opt.history['loss']
    aciertos_val_por_optimizador[nombre] = historial_opt.history['val_accuracy']
    print(f"{nombre}: entrenada")

Las curvas de los cuatro optimizadores. El eje de arriba cuenta las actualizaciones de los pesos.

In [ ]:
def epoca_a_actualizacion(epoca):
    return epoca * actualizaciones_por_epoca


def actualizacion_a_epoca(actualizacion):
    return actualizacion / actualizaciones_por_epoca


fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
for numero_opt in range(len(nombres_optimizadores)):
    nombre = nombres_optimizadores[numero_opt]
    color = f'C{numero_opt}'
    ax[0].plot(epocas, perdidas_por_optimizador[nombre], color=color, label=nombre)
    ax[1].plot(epocas, aciertos_val_por_optimizador[nombre], color=color, label=nombre)

ax[0].set_yscale('log')
ax[0].set_ylabel('pérdida de entrenamiento (escala logarítmica)')
ax[0].set_title('Pérdida de entrenamiento')
ax[1].set_ylabel('exactitud en validación')
ax[1].set_title('Exactitud en validación')
for eje in ax:
    eje.set_xlabel('época')
    eje.grid(alpha=0.3)
    eje.legend()
    eje_superior = eje.secondary_xaxis('top', functions=(epoca_a_actualizacion, actualizacion_a_epoca))
    eje_superior.set_xlabel('actualizaciones de los pesos')
plt.tight_layout()
plt.show()

Las mismas curvas en cifras: pérdida de entrenamiento y exactitud de validación en algunas épocas.

In [ ]:
filas_tabla = []
for nombre in nombres_optimizadores:
    perdidas = perdidas_por_optimizador[nombre]
    aciertos_val = aciertos_val_por_optimizador[nombre]
    fila = {
        'optimizador': nombre,
        'pérdida train, época 1': perdidas[0],
        f'pérdida train, época {n_epocas}': perdidas[-1],
        'exactitud val., época 5': aciertos_val[4],
        f'exactitud val., época {n_epocas}': aciertos_val[-1],
    }
    filas_tabla.append(fila)

tabla_optimizadores = pd.DataFrame(filas_tabla)
tabla_optimizadores = tabla_optimizadores.set_index('optimizador')
tabla_optimizadores.round(4)

**Pregunta 4.2.** ¿Qué optimizadores reducen antes la pérdida de entrenamiento? ¿Cuál termina con mejor
exactitud de validación?

**Tu respuesta:** *escribe aquí*

**Pregunta 4.3.** ¿Ha terminado de aprender SGD sin momento en las 30 épocas? ¿Qué cambiarías para que
aprendiera más deprisa? ¿Puedes concluir de este experimento que SGD es peor optimizador que los otros?

**Tu respuesta:** *escribe aquí*

## 5. Regresión: estimar el ángulo de giro

Cada imagen trae también el ángulo que se giró, un número entero entre −45° y 45° (columna `angle` de
los CSV). Estimar un número en lugar de elegir una clase es un problema de **regresión**, y se resuelve
con el mismo tipo de red: las entradas (los 784 píxeles) y las capas ocultas no cambian. Cambian la capa
de salida, la pérdida y lo que medimos:

| | Clasificación (secciones 2 a 4) | Regresión (esta sección) |
|---|---|---|
| Capa de salida | `Dense(10, activation='softmax')` | `Dense(1)`: una neurona, sin activación |
| Pérdida | `'sparse_categorical_crossentropy'` | `'mse'`: error cuadrático medio |
| Métricas | exactitud (`'accuracy'`) | error absoluto medio, raíz del error cuadrático medio y R² |

La referencia es predecir siempre el ángulo medio de las imágenes de entrenamiento. En esta sección solo
usamos entrenamiento y validación. La celda siguiente lee los ángulos, repite la partición de la sección 1 con
las imágenes y sus ángulos, y calcula la referencia.

Variables de esta sección:

| Variable | Qué contiene |
|---|---|
| `angulo_train_completo` | ángulo de cada una de las 5.000 imágenes de entrenamiento |
| `X_train`, `X_val`, `angulo_train`, `angulo_val` | imágenes y ángulos de entrenamiento y de validación (la partición de la sección 1) |
| `angulo_medio`, `error_media_val` | ángulo medio de entrenamiento y error absoluto medio en validación si siempre lo predecimos |
| `perdida_regresion` | pérdida para estimar un número: el error cuadrático medio |
| `modelo_regresion` | MLP con una capa oculta de 64 neuronas y salida de regresión |
| `modelo_regresion_profundo` | MLP con dos capas ocultas, de 128 y 64 neuronas, y salida de regresión |
| `metricas_64`, `metricas_profundo` | pérdida y métricas de cada red en validación, en un diccionario |
| `tabla_metricas_regresion` | MAE, RMSE y R² en validación de la referencia y de las dos redes |

In [ ]:
ruta_csv_train = os.path.join(carpeta_digitos, 'digitTrain.csv')
tabla_train_completo = pd.read_csv(ruta_csv_train)
angulo_train_completo = tabla_train_completo['angle'].to_numpy()

# Misma partición que en la sección 1 (mismos test_size, random_state y stratify): X_train y X_val
# salen iguales que allí, y cada ángulo queda junto a su imagen.
X_train, X_val, angulo_train, angulo_val = train_test_split(
    X_train_completo, angulo_train_completo, test_size=1000, random_state=0, stratify=y_train_completo)

angulo_medio = angulo_train.mean()
errores_media_val = np.abs(angulo_val - angulo_medio)
error_media_val = errores_media_val.mean()
print(f"Ángulo medio de entrenamiento: {angulo_medio:.2f}°")
print(f"Predecir siempre el ángulo medio: error absoluto medio en validación = {error_media_val:.1f}°")

### Una capa oculta

Empezamos con la red de la sección 2, con una capa oculta de 64 neuronas, cambiando la capa de salida y
la pérdida.

In [ ]:
keras.utils.set_random_seed(0)

modelo_regresion = keras.Sequential([
    keras.Input(shape=(784,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(1),
])

Elegimos la pérdida y compilamos la red. Esta vez pedimos tres métricas, las de regresión de la semana
2: el error absoluto medio (`'mae'`), la raíz del error cuadrático medio (`'root_mean_squared_error'`),
que pesa más los errores grandes, y el coeficiente de determinación R² (`'r2_score'`), que vale 0 si la
predicción no mejora a predecir siempre la media y 1 si es perfecta. Las dos primeras están en grados.

In [ ]:
perdida_regresion = 'mse'

modelo_regresion.compile(optimizer='adam', loss=perdida_regresion,
                         metrics=['mae', 'root_mean_squared_error', 'r2_score'])
modelo_regresion.summary()

La entrenamos con parada temprana, como al final de la sección 2: un máximo de 200 épocas y paciencia
10. La celda imprime también las claves del historial: ahora son la pérdida y las tres métricas, cada
una en entrenamiento y en validación. Tarda unos segundos.

In [ ]:
parada_regresion = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
historial_regresion = modelo_regresion.fit(X_train, angulo_train, validation_data=(X_val, angulo_val),
                                           epochs=200, batch_size=32, callbacks=[parada_regresion], verbose=0)
n_epocas_regresion = len(historial_regresion.history['loss'])
print(f"MLP de 64 neuronas: se detuvo en la época {n_epocas_regresion}")

claves_regresion = list(historial_regresion.history.keys())
print("Claves del historial:", claves_regresion)

### Varias capas ocultas

Como en clasificación, para tener más capas ocultas basta con poner más capas `Dense` en la lista.
Esta red tiene dos capas ocultas con ReLU, de 128 y 64 neuronas, y la salida de regresión.

In [ ]:
keras.utils.set_random_seed(0)

modelo_regresion_profundo = keras.Sequential([
    keras.Input(shape=(784,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(1),
])
modelo_regresion_profundo.compile(optimizer='adam', loss='mse',
                                  metrics=['mae', 'root_mean_squared_error', 'r2_score'])
modelo_regresion_profundo.summary()

La entrenamos igual, con parada temprana. Tarda unos segundos.

In [ ]:
parada_profundo = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
historial_regresion_profundo = modelo_regresion_profundo.fit(
    X_train, angulo_train, validation_data=(X_val, angulo_val),
    epochs=200, batch_size=32, callbacks=[parada_profundo], verbose=0)
n_epocas_profundo = len(historial_regresion_profundo.history['loss'])
print(f"MLP de dos capas: se detuvo en la época {n_epocas_profundo}")

### Métricas de regresión

Comparamos las dos redes con la referencia en validación. Las métricas de las redes las da `evaluate`,
como en clasificación. Con varias métricas conviene `return_dict=True`: `evaluate` devuelve entonces un
diccionario con la pérdida y cada métrica por su nombre, el mismo que en el historial. Sin él devuelve
una lista en el orden de `compile`, y es fácil equivocarse al repartirla en variables. La referencia no
es una red, así que sus métricas se calculan con sklearn, como en la semana 2.

In [ ]:
metricas_64 = modelo_regresion.evaluate(X_val, angulo_val, verbose=0, return_dict=True)
metricas_profundo = modelo_regresion_profundo.evaluate(X_val, angulo_val, verbose=0, return_dict=True)

prediccion_media = np.full(len(angulo_val), angulo_medio)

fila_media = {
    'modelo': 'predecir siempre el ángulo medio',
    'MAE (°)': mean_absolute_error(angulo_val, prediccion_media),
    'RMSE (°)': root_mean_squared_error(angulo_val, prediccion_media),
    'R²': r2_score(angulo_val, prediccion_media),
}
fila_64 = {
    'modelo': 'MLP de 64 neuronas',
    'MAE (°)': metricas_64['mae'],
    'RMSE (°)': metricas_64['root_mean_squared_error'],
    'R²': metricas_64['r2_score'],
}
fila_profundo = {
    'modelo': 'MLP de dos capas (128 y 64)',
    'MAE (°)': metricas_profundo['mae'],
    'RMSE (°)': metricas_profundo['root_mean_squared_error'],
    'R²': metricas_profundo['r2_score'],
}
tabla_metricas_regresion = pd.DataFrame([fila_media, fila_64, fila_profundo])
tabla_metricas_regresion = tabla_metricas_regresion.set_index('modelo')
tabla_metricas_regresion.round(3)

## Resumen

- Un **MLP** para imágenes recibe cada imagen como una fila de píxeles (`reshape`), la pasa por capas
  `Dense` con ReLU y termina en una capa softmax con una neurona por clase; con etiquetas enteras, la
  pérdida es `sparse_categorical_crossentropy`.
- Para tener **varias capas ocultas** basta con poner más capas `Dense` en la lista de `Sequential`.
- La **pérdida** es lo que se minimiza; las **métricas** solo se miden. Se piden en `compile`, una o
  varias; `fit` guarda cada una en el historial, época a época, en entrenamiento y en validación
  (`val_`), y `evaluate` las devuelve junto a la pérdida. La matriz de confusión y
  `classification_report` dan las métricas por clase.
- La **parada temprana** (`EarlyStopping`) deja que la validación decida cuántas épocas entrenar.
- La **regularización** frena el sobreentrenamiento: L2 en una capa (`kernel_regularizer`), capas
  `Dropout` y la propia parada temprana.
- Las **curvas** de pérdida y exactitud por época, en entrenamiento y en validación, muestran cómo
  aprende la red.
- Los hiperparámetros, como el número de neuronas, se eligen con **validación**; el modelo elegido se
  reentrena con entrenamiento + validación y el **test** se mide una sola vez.
- Una **época** recorre todos los datos de entrenamiento; con lotes de tamaño B, en ella hay n/B
  **actualizaciones** de los pesos.
- Para comparar **optimizadores** se usan la misma red, la misma semilla y el mismo número de épocas, y
  se dice qué tasa de aprendizaje usa cada uno.
- La misma red sirve para **regresión**: capa de salida `Dense(1)` sin activación, pérdida `'mse'` y, en
  lugar de la exactitud, las métricas de regresión: MAE, RMSE y R².

En `Semana5_convolucionales` se entrena una red convolucional con las mismas imágenes y la misma
partición, y se compara con la exactitud de test del MLP de este cuaderno.